In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (private adapter dataset may fail):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
adapter_dirs = {}
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    v = variant.split("_")[0]
    cands = sorted(glob.glob(f"/kaggle/input/**/{v}/adapter", recursive=True))
    hits = [c for c in cands if (Path(c) / "adapter_config.json").exists()]
    if hits:
        adapter_dirs[variant] = hits[0]
        print(f"adapter [{variant}] -> {hits[0]}")
    else:
        print(f"!! NO adapter for [{variant}]")
if not adapter_dirs:
    print("INPUT DIRS:", os.listdir("/kaggle/input"))
    raise SystemExit("no adapters found")
open("/kaggle/working/adapters.json","w").write(json.dumps(adapter_dirs, indent=2))


In [ ]:
import json, subprocess, sys
adapters = json.load(open("/kaggle/working/adapters.json"))
os.makedirs("/kaggle/output/traj", exist_ok=True)
os.makedirs("/kaggle/output/sweep", exist_ok=True)

# One collect pass per (variant, task_set). Each writes a trajectory JSONL.
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    ad = f"{variant}={adapters[variant]}"
    for task_set, tasks in (("calib", "arc_easy,arc_challenge,hellaswag,piqa,boolq,sciq"), ("bench", "arc_easy,arc_challenge,hellaswag,piqa,boolq,sciq")):
        out = f"/kaggle/output/traj/{variant}-{task_set}.jsonl"
        if os.path.exists(out) and os.path.getsize(out) > 0:
            print("skip (exists):", out, flush=True)
            continue
        print(f">>> collect {variant} {task_set}", flush=True)
        subprocess.run([sys.executable, "scripts/calibrate_halt.py", "collect",
                        "--base", "/kaggle/working/jetmoe-8b",
                        "--model", variant, "--adapters", ad,
                        "--task_set", task_set, "--tasks", tasks,
                        "--limit", "60", "--max_loops", "8",
                        "--out", out],
                       cwd="/kaggle/working/arc", check=True)

# Offline replay: the whole grid, no GPU needed.
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    for task_set in ("calib", "bench"):
        traj = f"/kaggle/output/traj/{variant}-{task_set}.jsonl"
        if not os.path.exists(traj):
            print("MISSING", traj, flush=True); continue
        print(f">>> sweep {variant} {task_set}", flush=True)
        subprocess.run([sys.executable, "scripts/calibrate_halt.py", "sweep",
                        "--traj", traj,
                        "--out", f"/kaggle/output/sweep/{variant}-{task_set}-sweep.csv",
                        "--caps", "2,3,4,6,8",
                        "--biases", "0.20,0.30,0.40,0.50,0.60,0.70",
                        "--ks", "4,8,14,24",
                        "--thresholds", "0.30,0.45,0.60,0.75",
                        "--min_gains", "0.0,0.02,0.04,0.07"],
                       cwd="/kaggle/working/arc", check=True)
print("ALL DONE")
